**Question 1**

In [ ]:
import pandas as pd
import requests
from io import StringIO
import re

In [ ]:
# Data Loading (from iposcoop.com)

url = "https://www.iposcoop.com/ipos-recently-filed"
 
headers = {
    'User-Agent': (
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/91.0.4472.124 Safari/537.36'
    )
}
 
try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    # Use StringIO buffer
    df_ipos = pd.read_html(StringIO(response.text))[0]
    print(f"Successfully loaded IPO data. Shape: {df_ipos.shape}")
    print(f"\nColumn names:\n{df_ipos.columns.tolist()}\n")
    
except Exception as e:
    print(f"Error loading data: {e}")
    exit()

In [ ]:
# filter withdrawn IPOs

withdrawn_df = df_ipos[df_ipos['Expected To Trade'] == 'Withdrawn'].copy()
print(f"Number of withdrawn IPOs: {len(withdrawn_df)}")
withdrawn_df.head()

In [ ]:
# Company Classification

def classify_company_type(company_name):
    """
    Categorize company based on patterns (order matters).
    Returns the first matched category.
    """
    if pd.isna(company_name):
        return 'Other'
    
    name = str(company_name)
    
    # Order matters: check in this sequence
    if 'Technologies' in name:
        return 'Technologies'
    elif 'Acquisition Corp' in name or 'Acquisition Corporation' in name or 'Corp' in name:
        return 'Acquisition Corp'
    elif 'Inc' in name or 'Incorporated' in name:
        return 'Inc.'
    elif 'Group' in name:
        return 'Group'
    elif 'Ltd' in name or 'Limited' in name:
        return 'Limited'
    elif 'Holdings' in name or 'Holding' in name:
        return 'Holdings'
    else:
        return 'Other'
 
withdrawn_df['Company Type'] = withdrawn_df['Company'].apply(classify_company_type)
print(f"\nCompany Type distribution:")
print(withdrawn_df['Company Type'].value_counts())

In [ ]:
# Price Parsing

def extract_numeric(price_str):
    """
    Extract numeric value from strings like '$8.00' or '-'
    Returns float or NaN for missing values
    """
    if pd.isna(price_str):
        return None
    
    price_str = str(price_str).strip()
    
    # Handle missing values
    if price_str == '-' or price_str == '':
        return None
    
    # Remove all non-numeric characters except decimal point
    cleaned = re.sub(r'[^\d.]', '', price_str)
    
    try:
        return float(cleaned) if cleaned else None
    except ValueError:
        return None

withdrawn_df['Price_Low'] = withdrawn_df['Price Low'].apply(extract_numeric)
withdrawn_df['Price_High'] = withdrawn_df['Price High'].apply(extract_numeric)
withdrawn_df['Avg_price'] = (withdrawn_df['Price_Low'] + withdrawn_df['Price_High']) / 2.0

withdrawn_df[['Company', 'Price Low', 'Price High', 'Avg_price']].head(50)

In [ ]:
# Numeric Conversion

withdrawn_df['Shares_millions'] = pd.to_numeric(
    withdrawn_df['Shares (millions)'].astype(str).str.replace('$', '').str.replace(',', ''),
    errors='coerce'
)

withdrawn_df['Est_Vol_millions'] = pd.to_numeric(
    withdrawn_df['Est $ Vol (millions)'].astype(str).str.replace('$', '').str.replace(',', ''),
    errors='coerce'
)

In [ ]:
# Value Calculation

withdrawn_df['Shares_offered_value'] = (withdrawn_df['Shares_millions'] * withdrawn_df['Avg_price']).fillna(withdrawn_df['Est_Vol_millions'])

In [ ]:
# Aggregation
result = withdrawn_df.groupby('Company Type')['Shares_offered_value'].sum().sort_values(ascending=False)

print("="*60)
print("WITHDRAWAL VALUES BY COMPANY TYPE (in $ millions)")
print("="*60)
print(result)
print()
print(f"Answer: {result.idxmax()} - ${result.max():,.2f} million")

**Qeustion 2**

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from io import StringIO
import requests

In [ ]:
# Data Loading

url = "https://www.iposcoop.com/2025-pricings/"

headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'}
response = requests.get(url, headers=headers)
df_ipos_2025 = pd.read_html(StringIO(response.text))[0]

print(f"Total IPOs 2025: {len(df_ipos_2025)}")

In [ ]:
# Filtering

df_ipos_2025['Offer Date'] = pd.to_datetime(df_ipos_2025['Offer Date'], errors='coerce')
df_ipos_2025['Return'] = pd.to_numeric(df_ipos_2025['Return'].astype(str).str.replace('%', ''), errors='coerce')

filtered_ipos = df_ipos_2025[
    (df_ipos_2025['Offer Date'] < '2025-09-01') & 
    (df_ipos_2025['Return'] != 0)
].copy()

print(f"Filtered IPOs (before Sept 1, 2025, non-zero return): {len(filtered_ipos)}")
print(filtered_ipos[['Symbol', 'Offer Date', 'Return']].head())

In [ ]:
# Data Download

tickers = filtered_ipos['Symbol'].tolist()
stocks_df = pd.DataFrame()

for ticker in tickers:
    try:
        data = yf.download(ticker, start='2024-09-01', end='2026-09-12', progress=False)
        if len(data) > 0:
            # Extract Close column for this ticker
            close_data = data[('Close', ticker)].reset_index()
            close_data.columns = ['Date', 'Close']
            close_data['Ticker'] = ticker
            stocks_df = pd.concat([stocks_df, close_data], ignore_index=True)
    except:
        pass

print(f"Downloaded stocks: {stocks_df['Ticker'].nunique()}")
print(stocks_df.head())

In [ ]:
# Feature Engineering

stocks_df.columns.name = None

stocks_df['Close'] = pd.to_numeric(stocks_df['Close'], errors='coerce')
stocks_df['growth_252d'] = stocks_df.groupby('Ticker')['Close'].transform(lambda x: x / x.shift(252))
stocks_df['volatility'] = stocks_df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(30).std() * np.sqrt(252))

print(stocks_df[['Ticker', 'growth_252d', 'volatility']].tail(10))

In [ ]:
# Sharpe Ratio Calculation

stocks_df['Sharpe'] = (stocks_df['growth_252d'] - 0.05) / stocks_df['volatility']

print(stocks_df[['Ticker', 'growth_252d', 'volatility', 'Sharpe']].tail())

In [ ]:
# Final Analysis

stocks_df['Date'] = pd.to_datetime(stocks_df['Date'])
final_df = stocks_df[stocks_df['Date'] == '2026-09-11'].copy()

print(f"\nStocks on 2026-09-11: {len(final_df)}")
print("\nSharpe Ratio statistics:")
print(final_df['Sharpe'].describe())
print(f"\nMedian Sharpe Ratio: {final_df['Sharpe'].median()}")

In [ ]:
# Compare median vs. mean for growth_252d
print("Growth (252d) Statistics:")
print(f"Mean growth_252d: {final_df['growth_252d'].mean():.4f}")
print(f"Median growth_252d: {final_df['growth_252d'].median():.4f}")
print(f"Difference (mean - median): {final_df['growth_252d'].mean() - final_df['growth_252d'].median():.4f}")
print(f"Max growth: {final_df['growth_252d'].max():.4f}")
print(f"Min growth: {final_df['growth_252d'].min():.4f}")

There's significant positive skew in growth returns, with the mean (1.06x) substantially higher than the median (0.59x), a difference of 0.46. This indicates that a handful of high-growth outliers are pulling the average well above what a typical IPO experienced. The maximum growth of 33.64x (MGRT) significantly inflates the mean, suggesting that most IPOs performed more modestly.

In [ ]:
# Count of stocks that reached 252-day milestone
stocks_with_252d = final_df['growth_252d'].notna().sum()
print(f"\nStocks with 252-day data: {stocks_with_252d} out of {len(final_df)}")

Nearly all stocks reached the 252-day trading milestone, with 130 out of 132 stocks (98.5%) successfully accumulating a full year of price data. Only 2 stocks delisted before completing the period.

In [ ]:
# Compare Sharpe ratio vs growth

comparison = final_df[['Ticker', 'growth_252d', 'Sharpe']].dropna().sort_values('Sharpe', ascending=False)
print("\nTop 10 by Sharpe Ratio:")
print(comparison.head(10))
print("\nTop 10 by Growth:")
print(comparison.sort_values('growth_252d', ascending=False).head(10))

Risk-adjusted returns (Sharpe ratio) do not track with pure growth metrics. The highest-growth stock (MGRT with 33.64x returns) has a relatively poor Sharpe ratio of 0.26, indicating high volatility relative to returns. Conversely, stocks with the highest Sharpe ratios often show more modest growth rates. For example, HCMAU achieved only 1.03x growth but a Sharpe ratio of 5.39, indicating much better risk-adjusted performance. This demonstrates that controlling for volatility reveals a different ranking of stocks than growth alone, and that lower-volatility positions can deliver more attractive risk-adjusted returns even with lower nominal growth.

**Question 3**

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from io import StringIO
import requests

In [ ]:
# Data Loading

url = "https://www.iposcoop.com/2025-pricings/"

headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'}
response = requests.get(url, headers=headers)
df_ipos_2025 = pd.read_html(StringIO(response.text))[0]

print(f"Total IPOs 2025: {len(df_ipos_2025)}")

# Filtering

df_ipos_2025['Offer Date'] = pd.to_datetime(df_ipos_2025['Offer Date'], errors='coerce')
df_ipos_2025['Return'] = pd.to_numeric(df_ipos_2025['Return'].astype(str).str.replace('%', ''), errors='coerce')

filtered_ipos = df_ipos_2025[
    (df_ipos_2025['Offer Date'] < '2025-09-01') & 
    (df_ipos_2025['Return'] != 0)
].copy()

print(f"Filtered IPOs (before Sept 1, 2025, non-zero return): {len(filtered_ipos)}")
print(filtered_ipos[['Symbol', 'Offer Date', 'Return']].head())

# Data Download

tickers = filtered_ipos['Symbol'].tolist()
stocks_df = pd.DataFrame()

for ticker in tickers:
    try:
        data = yf.download(ticker, start='2024-09-01', end='2026-09-12', progress=False)
        if len(data) > 0:
            # Extract Close column for this ticker
            close_data = data[('Close', ticker)].reset_index()
            close_data.columns = ['Date', 'Close']
            close_data['Ticker'] = ticker
            stocks_df = pd.concat([stocks_df, close_data], ignore_index=True)
    except:
        pass

print(f"Downloaded stocks: {stocks_df['Ticker'].nunique()}")
print(stocks_df.head())

In [ ]:
# Calculate future growth for 1-12 months (21 trading days per month)
for month in range(1, 13):
    days = month * 21
    stocks_df[f'future_growth_{month}_m'] = stocks_df.groupby('Ticker')['Close'].transform(
        lambda x: x.shift(-days) / x
    )

print("Future growth columns created")
print(stocks_df.columns)

In [ ]:
# Identify IPO entry points (first trading day for each ticker)
ipo_entry = stocks_df.groupby('Ticker').agg({
    'Date': 'min'
}).reset_index()
ipo_entry.columns = ['Ticker', 'IPO_Date']

print(f"IPO entry points identified for {len(ipo_entry)} stocks")
print(ipo_entry.head())

In [ ]:
# Join to get returns from IPO date
ipo_returns = stocks_df.merge(
    ipo_entry,
    left_on=['Ticker', 'Date'],
    right_on=['Ticker', 'IPO_Date'],
    how='inner'
)

print(f"IPO return records: {len(ipo_returns)}")
print(ipo_returns.head())

In [ ]:
# Calculate median growth for each holding period
growth_cols = [f'future_growth_{m}_m' for m in range(1, 13)]
stats = ipo_returns[growth_cols].describe()

print("\nGrowth Statistics by Holding Period:")
print(stats)

# Find optimal month
median_row = ipo_returns[growth_cols].median()
optimal_month = int(median_row.idxmax().split('_')[2])
optimal_value = median_row.max()

print(f"\nOptimal holding period: {optimal_month} months")
print(f"Median growth: {optimal_value:.4f}x")

The analysis reveals a clear and consistent downward trend in median returns as the holding period extends from 1 to 12 months. The optimal holding period is 1 month, with a median growth of 0.9295x, representing a nearly 7% loss for the typical IPO holder. Each subsequent month sees median returns decline further, bottoming at 0.4918x (approximately 51% loss) at the 12-month mark. This suggests that most 2025 IPOs underperformed significantly in their first year of trading.

In [ ]:
# Comparison: Median vs Mean
print("\nMedian vs Mean by Month:")
comparison = pd.DataFrame({
    'Month': range(1, 13),
    'Median': [ipo_returns[f'future_growth_{m}_m'].median() for m in range(1, 13)],
    'Mean': [ipo_returns[f'future_growth_{m}_m'].mean() for m in range(1, 13)]
})
comparison['Difference'] = comparison['Mean'] - comparison['Median']
print(comparison)

While median returns decay steadily from month 1 to month 12, the mean values become increasingly distorted by extreme outliers starting in month 5, with the mean reaching 77x in month 5 and 51x in month 10, while the corresponding medians remain around 0.7x and 0.53x respectively. These astronomical mean values indicate the presence of a small number of extreme gainers (likely delisted or bankrupt stocks showing data anomalies) that completely distort the average.

**Question 4**

In [1]:
# Data Acquisition

import gdown
import pandas as pd

file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)
df = pd.read_parquet("data.parquet", engine="fastparquet")

print(f"Data shape: {df.shape}")
print(df.columns)

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=1c41846c-1fb9-4563-92f8-0d2584031f8e
To: /Users/rob/Projects/GitHub/stock-markets-analytics-zoomcamp/02-dataframe-analysis/data.parquet
100%|██████████| 130M/130M [00:03<00:00, 37.7MB/s] 


Data shape: (229932, 203)
Index(['Open', 'High', 'Low', 'Close_x', 'Volume', 'Dividends', 'Stock Splits',
       'Ticker', 'Year', 'Month',
       ...
       'growth_brent_oil_7d', 'growth_brent_oil_30d', 'growth_brent_oil_90d',
       'growth_brent_oil_365d', 'growth_btc_usd_1d', 'growth_btc_usd_3d',
       'growth_btc_usd_7d', 'growth_btc_usd_30d', 'growth_btc_usd_90d',
       'growth_btc_usd_365d'],
      dtype='object', length=203)


In [3]:
print(df.columns.tolist())
print(df.head())


['Open', 'High', 'Low', 'Close_x', 'Volume', 'Dividends', 'Stock Splits', 'Ticker', 'Year', 'Month', 'Weekday', 'Date', 'growth_1d', 'growth_3d', 'growth_7d', 'growth_30d', 'growth_90d', 'growth_365d', 'growth_future_30d', 'SMA10', 'SMA20', 'growing_moving_average', 'high_minus_low_relative', 'volatility', 'is_positive_growth_30d_future', 'ticker_type', 'index_x', 'adx', 'adxr', 'apo', 'aroon_1', 'aroon_2', 'aroonosc', 'bop', 'cci', 'cmo', 'dx', 'macd', 'macdsignal', 'macdhist', 'macd_ext', 'macdsignal_ext', 'macdhist_ext', 'macd_fix', 'macdsignal_fix', 'macdhist_fix', 'mfi', 'minus_di', 'mom', 'plus_di', 'dm', 'ppo', 'roc', 'rocp', 'rocr', 'rocr100', 'rsi', 'slowk', 'slowd', 'fastk', 'fastd', 'fastk_rsi', 'fastd_rsi', 'trix', 'ultosc', 'willr', 'index_y', 'ad', 'adosc', 'obv', 'atr', 'natr', 'ht_dcperiod', 'ht_dcphase', 'ht_phasor_inphase', 'ht_phasor_quadrature', 'ht_sine_sine', 'ht_sine_leadsine', 'ht_trendmod', 'avgprice', 'medprice', 'typprice', 'wclprice', 'index', 'cdl2crows', '

In [4]:
# Strategy Setup and Filtering

df['Date'] = pd.to_datetime(df['Date'])
filtered_df = df[
    (df['Date'] >= '2000-01-01') & 
    (df['Date'] <= '2025-06-01') &
    (df['rsi'] < 30)
].copy()

print(f"Number of oversold signals: {len(filtered_df)}")
print(f"Date range: {filtered_df['Date'].min()} to {filtered_df['Date'].max()}")

Number of oversold signals: 5206
Date range: 2000-01-27 00:00:00 to 2025-04-22 00:00:00


In [5]:
# Calculate profit
net_income = 1000 * (filtered_df['growth_future_30d'] - 1).sum()
net_income_thousands = net_income / 1000

print(f"\nTotal net income: ${net_income:,.2f}")
print(f"Net income (in $ thousands): ${net_income_thousands:,.2f}k")


Total net income: $65,805.59
Net income (in $ thousands): $65.81k


In [6]:
# Observations
print(f"\nTotal trades: {len(filtered_df)}")
print(f"Average 30-day return: {filtered_df['growth_future_30d'].mean() - 1:.4f} ({(filtered_df['growth_future_30d'].mean() - 1)*100:.2f}%)")
print(f"Win rate: {(filtered_df['growth_future_30d'] > 1).sum() / len(filtered_df) * 100:.2f}%")


Total trades: 5206
Average 30-day return: 0.0126 (1.26%)
Win rate: 55.13%


**Question 5**

Proposed Strategies to Improve IPO Profitability:

1. Sector/Industry Filtering:
IPOs in growth sectors (technology, biotech) historically outperform traditional sectors (energy, materials). Filter the IPO list by sector and analyze performance separately. Exclude defensive/utility IPOs that tend to underperform.

2. Market Timing:
IPOs perform better when the broader market (S&P 500, Nasdaq) is in an uptrend. Add a filter: only trade IPOs when the market's 30-day growth is positive. Avoid IPO investing during market downturns.

3. Lock-Up Period Exploitation:
Many IPOs drop significantly when lock-up periods expire (typically 180 days). Short stocks 30 days before lock-up expiration, or avoid holding through those dates entirely.

4. Company Size/IPO Valuation:
Smaller IPOs (lower market cap at listing) tend to be more volatile with higher upside potential. Compare median returns across IPO size buckets to identify the sweet spot.

5. Technical Indicators (Beyond RSI):
Combine RSI < 30 (oversold) with other momentum indicators like MACD, Bollinger Bands, or Stochastic to reduce false signals and improve win rate.

6. Profitability/Unit Economics:
Prioritize IPOs from profitable companies with positive cash flow over high-growth money-losing startups. Screen for positive earnings or positive free cash flow at IPO.

7. IPO Quiet Period Momentum:
Track pre-IPO momentum and investor demand (oversubscription ratios). IPOs with strong demand often have positive early returns; those with weak demand underperform.

8. Mean Reversion Strategy:
If most IPOs drop on day 1, buy them on day 3-5 when they're oversold, rather than on listing day.

9. Macro Conditions Filter:
Only invest in IPOs when Fed policy is dovish (rates falling) and credit spreads are narrow. Avoid IPO investing when rates are rising sharply.

10. Combine with Macro Hedges:
Use gold, bonds, or puts on broad indices to hedge IPO portfolio risk, improving risk-adjusted returns even if absolute returns remain modest.

For submission:

Filter IPOs by sector, focusing exclusively on growth industries (technology, biotech) while excluding defensive sectors that historically underperform. Only execute IPO trades when the broader market (S&P 500, Nasdaq) is in an uptrend, indicated by positive 30-day returns, to avoid fighting headwinds from market downturns. Avoid holding IPOs through their lock-up expiration periods (typically 180 days), as these dates frequently trigger sharp selloffs—either exit before lock-up ends or short the stock 30 days prior to capitalize on the expected decline.